# Quantathon — T4 GPU, FastAPI, Cloudflare API và QAOA Circuit

Notebook này chạy trọn flow của project:

```text
Upload ZIP
→ cài dependencies
→ xác nhận CUDA-Q target nvidia
→ smoke test Qamomile → CUDA-Q
→ chạy tests
→ khởi động FastAPI
→ POST /api/runs
→ chạy các vòng ADMM + QAOA
→ in circuit của từng vòng ADMM
→ tạo Cloudflare API cho frontend local
```

Trước khi bắt đầu, chọn:

**Runtime → Change runtime type → T4 GPU**

Upload đúng file:

```text
Quantathon-Scaling-Benchmark-Isolated-Circuit-Fixed.zip
```

## 1. Kiểm tra T4 GPU

In [ ]:
import shutil
import subprocess

assert shutil.which("nvidia-smi"), (
    "Không tìm thấy GPU. Hãy chọn Runtime → Change runtime type → T4 GPU."
)

subprocess.run(["nvidia-smi"], check=True)


## 2. Upload và giải nén project

In [ ]:
from google.colab import files
from pathlib import Path
import os
import shutil
import zipfile

# Luôn rời khỏi project cũ trước khi xóa/giải nén lại.
os.chdir("/content")

print("Upload Quantathon-Scaling-Benchmark-Isolated-Circuit-Fixed.zip")
uploaded = files.upload()

zip_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]
assert zip_names, "Bạn chưa upload file ZIP."

zip_path = Path("/content") / Path(zip_names[0]).name
assert zip_path.exists(), f"Không tìm thấy ZIP sau upload: {zip_path}"

extract_root = Path("/content/quantathon_project")
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_root)

requirements_matches = list(
    extract_root.rglob("backend/requirements-quantum-colab.txt")
)
assert requirements_matches, (
    "ZIP không chứa backend/requirements-quantum-colab.txt"
)

REQ_FILE = requirements_matches[0].resolve()
PROJECT_DIR = REQ_FILE.parent.parent
BACKEND_DIR = PROJECT_DIR / "backend"
HYBRID_FILE = BACKEND_DIR / "app/backends/hybrid_qaoa.py"
PIPELINE_FILE = BACKEND_DIR / "app/services/pipeline.py"

os.chdir(PROJECT_DIR)

print("ZIP:", zip_path)
print("PROJECT_DIR:", PROJECT_DIR)
print("BACKEND_DIR:", BACKEND_DIR)
print("Requirements:", REQ_FILE)


## 3. Xác nhận ZIP đã có phần xuất circuit đúng

In [ ]:
hybrid_code = HYBRID_FILE.read_text(encoding="utf-8")
pipeline_code = PIPELINE_FILE.read_text(encoding="utf-8")

assert "def _save_and_print_cudaq_circuit(" in hybrid_code, (
    "Bạn đang dùng ZIP cũ, chưa có circuit exporter."
)
assert 'qubo.metadata["admm_round"] = round_index + 1' in pipeline_code, (
    "pipeline.py chưa gắn số vòng ADMM vào QUBO."
)
assert (
    'from app.qubo.builder import build_dynamic_qubo\n'
    'qubo.metadata["admm_round"]'
) not in pipeline_code, (
    "pipeline.py vẫn còn dòng metadata sai vị trí ở phần import."
)

print("Project circuit integration: OK")


## 4. Cài dependencies

Các warning `Skipping cudaq ... not installed` trong runtime mới là bình thường.

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "uninstall", "-y",
        "cudaq",
        "cuda-quantum",
        "cuda-quantum-cu11",
        "cuda-quantum-cu12",
        "cuda-quantum-cu13",
    ],
    check=False,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip>=24"],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)],
    check=True,
)

print("Dependencies installed.")


## 5. Bắt buộc CUDA-Q chạy trên NVIDIA T4

In [ ]:
import os
import cudaq

os.environ["CUDAQ_TARGET"] = "nvidia"
os.environ["REQUIRE_CUDAQ"] = "1"
os.environ["PRINT_QAOA_CIRCUIT"] = "1"
os.environ["QAOA_CIRCUIT_DIR"] = "/content/qaoa-circuits"
os.environ["DRAW_QAOA_CIRCUIT_MAX_QUBITS"] = "16"

cudaq.set_target("nvidia")

target = cudaq.get_target()
target_name = getattr(target, "name", str(target))
print("CUDA-Q target:", target_name)
assert target_name == "nvidia", f"Target không phải nvidia: {target_name}"

@cudaq.kernel
def bell_state():
    q = cudaq.qvector(2)
    h(q[0])
    x.ctrl(q[0], q[1])
    mz(q)

counts = cudaq.sample(bell_state, shots_count=100)
print("Bell-state GPU smoke test:", counts)


## 6. Smoke test Qamomile → CUDA-Q của project

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(BACKEND_DIR)
env["CUDAQ_TARGET"] = "nvidia"
env["REQUIRE_CUDAQ"] = "1"

smoke_script = BACKEND_DIR / "scripts/quantum_smoke.py"
assert smoke_script.exists(), f"Không tìm thấy {smoke_script}"

subprocess.run(
    [sys.executable, str(smoke_script)],
    cwd=str(PROJECT_DIR),
    env=env,
    check=True,
)


## 7. Compile và chạy backend tests

In [ ]:
RUN_TESTS = True

subprocess.run(
    [sys.executable, "-m", "compileall", "-q", str(BACKEND_DIR / "app")],
    cwd=str(PROJECT_DIR),
    env=env,
    check=True,
)

if RUN_TESTS:
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q", str(BACKEND_DIR / "tests")],
        cwd=str(PROJECT_DIR),
        env=env,
        check=True,
    )
else:
    print("Đã bỏ qua tests.")


## 8. Khởi động FastAPI với circuit export bật

In [ ]:
import requests
import subprocess
import sys
import time
from pathlib import Path

subprocess.run(
    ["pkill", "-f", "uvicorn app.main:app"],
    check=False,
)

CIRCUIT_DIR = Path("/content/qaoa-circuits")
shutil.rmtree(CIRCUIT_DIR, ignore_errors=True)
CIRCUIT_DIR.mkdir(parents=True, exist_ok=True)

BACKEND_LOG = Path("/tmp/hquc-backend.log")
BACKEND_LOG.write_text("", encoding="utf-8")
backend_log_handle = BACKEND_LOG.open("a", encoding="utf-8")

backend_env = os.environ.copy()
backend_env.update({
    "PYTHONPATH": str(BACKEND_DIR),
    "PYTHONUNBUFFERED": "1",
    "CUDAQ_TARGET": "nvidia",
    "REQUIRE_CUDAQ": "1",
    "PRINT_QAOA_CIRCUIT": "1",
    "QAOA_CIRCUIT_DIR": str(CIRCUIT_DIR),
    "DRAW_QAOA_CIRCUIT_MAX_QUBITS": "16",
})

BACKEND_PROCESS = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "-m",
        "uvicorn",
        "app.main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd=str(BACKEND_DIR),
    env=backend_env,
    stdout=backend_log_handle,
    stderr=subprocess.STDOUT,
)

health_url = "http://127.0.0.1:8000/api/health"
last_error = None

for attempt in range(1, 61):
    if BACKEND_PROCESS.poll() is not None:
        print(BACKEND_LOG.read_text(encoding="utf-8", errors="replace"))
        raise RuntimeError("FastAPI process đã dừng.")

    try:
        response = requests.get(health_url, timeout=3)
        if response.ok:
            print("Backend PID:", BACKEND_PROCESS.pid)
            print("Health:", response.json())
            break
    except Exception as exc:
        last_error = exc

    time.sleep(1)
else:
    print(BACKEND_LOG.read_text(encoding="utf-8", errors="replace"))
    raise RuntimeError(f"Backend không sẵn sàng: {last_error}")


## 9. Chạy một full Hybrid request để sinh circuit

Cell này chạy thật:

```text
HiGHS baseline
→ LP relaxation
→ ADMM active block
→ Qamomile
→ CUDA-Q nvidia
→ Top-K reconstruction
```

Mặc định dùng 10 qubit để circuit còn đọc được.

In [ ]:
import json
import requests
import time

payload_path = PROJECT_DIR / "sample-run-request.json"
payload = json.loads(payload_path.read_text(encoding="utf-8"))

hybrid = payload.setdefault("hybrid_config", {})
hybrid.update({
    "qubit_budget": 10,
    "candidate_generators": 2,
    "candidate_hours": 5,
    "qaoa_depth": 1,
    "shots": 1000,
    "optimizer_shots": 256,
    "optimizer_evaluations": 20,
    "top_k": 10,
    "max_quantum_rounds": 3,
    "quantum_target": "nvidia",
    "allow_numpy_fallback": False,
})

# Xóa circuit cũ để chỉ hiển thị run này.
for path in CIRCUIT_DIR.glob("*"):
    path.unlink()

started = time.perf_counter()
response = requests.post(
    "http://127.0.0.1:8000/api/runs",
    json=payload,
    timeout=900,
)
elapsed = time.perf_counter() - started

print("HTTP status:", response.status_code)
if not response.ok:
    print(response.text)
    print("\nBACKEND LOG:\n", BACKEND_LOG.read_text(
        encoding="utf-8", errors="replace"
    )[-20000:])
    response.raise_for_status()

RUN_RESULT = response.json()
hybrid_result = RUN_RESULT["result"]["hybrid"]

print("Run ID:", RUN_RESULT["run_id"])
print("Elapsed:", round(elapsed, 3), "s")
print("ADMM rounds executed:", hybrid_result["round_count"])
print("Active qubits:", hybrid_result["active_qubits"])
print("Backend source:", hybrid_result["backend_source"])
print("Feasible:", hybrid_result["feasible"])
print("Operating cost:", hybrid_result["true_operating_cost"])

assert hybrid_result["backend_source"] == "qamomile_cudaq_nvidia"


## 10. Hiển thị circuit của từng vòng ADMM ngay trong Colab

In [ ]:
from IPython.display import HTML, Markdown, display
from pathlib import Path
import html
import json
import time

# Chờ filesystem flush.
for _ in range(20):
    if list(CIRCUIT_DIR.glob("*_circuit.txt")):
        break
    time.sleep(0.5)

ascii_files = sorted(CIRCUIT_DIR.glob("*_circuit.txt"))
source_files = sorted(CIRCUIT_DIR.glob("*_cudaq_kernel.py"))
error_files = sorted(CIRCUIT_DIR.glob("*_draw_error.txt"))

print("Circuit directory:", CIRCUIT_DIR)
print("Files:", [path.name for path in sorted(CIRCUIT_DIR.glob("*"))])

if ascii_files:
    for path in ascii_files:
        display(Markdown(f"### {path.stem}"))
        drawing = path.read_text(encoding="utf-8", errors="replace")
        display(HTML(
            "<div style='overflow-x:auto; border:1px solid #ddd; "
            "padding:12px; background:#fafafa'>"
            "<pre style='font-size:12px; line-height:1.15; "
            "white-space:pre; margin:0'>"
            + html.escape(drawing)
            + "</pre></div>"
        ))
else:
    print("Không có ASCII drawing. Hiển thị generated CUDA-Q source thay thế.")
    for path in error_files:
        print(path.name, ":", path.read_text(
            encoding="utf-8", errors="replace"
        ))
    for path in source_files:
        display(Markdown(f"### {path.stem}"))
        source = path.read_text(encoding="utf-8", errors="replace")
        display(HTML(
            "<div style='overflow-x:auto; border:1px solid #ddd; "
            "padding:12px; background:#fafafa'>"
            "<pre style='font-size:12px; white-space:pre; margin:0'>"
            + html.escape(source)
            + "</pre></div>"
        ))

assert ascii_files or source_files, (
    "Backend chưa xuất circuit/source. Hãy xem cell backend log phía dưới."
)


## 11. Xem mapping qubit → generator/hour

In [ ]:
import json
import pandas as pd
from IPython.display import display

metadata_files = sorted(CIRCUIT_DIR.glob("*_metadata.json"))

for path in metadata_files:
    metadata = json.loads(path.read_text(encoding="utf-8"))
    print(
        f"ADMM round {metadata['admm_round']} | "
        f"{metadata['qubit_count']} qubits"
    )
    display(pd.DataFrame(metadata["variable_mapping"]))


## 12. Kiểm tra circuit trong backend log

In [ ]:
log_text = BACKEND_LOG.read_text(
    encoding="utf-8",
    errors="replace",
)

marker = "QAOA CIRCUIT | ADMM ROUND"
if marker in log_text:
    print(log_text[log_text.find(marker):])
else:
    print(log_text[-20000:])


## 13. Tạo Cloudflare public API cho frontend

In [ ]:
from pathlib import Path
import re
import socket
import stat
import subprocess
import time
import urllib.request

cloudflared_path = Path("/content/cloudflared")

if not cloudflared_path.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64",
        cloudflared_path,
    )
    cloudflared_path.chmod(
        cloudflared_path.stat().st_mode | stat.S_IEXEC
    )

subprocess.run(
    ["pkill", "-f", str(cloudflared_path)],
    check=False,
)

TUNNEL_LOG = Path("/tmp/cloudflared.log")
TUNNEL_LOG.write_text("", encoding="utf-8")
tunnel_log_handle = TUNNEL_LOG.open("a", encoding="utf-8")

TUNNEL_PROCESS = subprocess.Popen(
    [
        str(cloudflared_path),
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate",
    ],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)

pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
PUBLIC_URL = None
last_error = None

for attempt in range(1, 121):
    if TUNNEL_PROCESS.poll() is not None:
        raise RuntimeError(
            "cloudflared đã dừng.\n"
            + TUNNEL_LOG.read_text(
                encoding="utf-8", errors="replace"
            )[-12000:]
        )

    text = TUNNEL_LOG.read_text(
        encoding="utf-8", errors="replace"
    )
    match = pattern.search(text)

    if match:
        candidate = match.group(0)
        hostname = candidate.removeprefix("https://")
        try:
            socket.getaddrinfo(hostname, 443)
            public_health = requests.get(
                f"{candidate}/api/health",
                timeout=10,
            )
            if public_health.ok:
                PUBLIC_URL = candidate
                break
            last_error = RuntimeError(
                f"HTTP {public_health.status_code}: "
                f"{public_health.text[:300]}"
            )
        except Exception as exc:
            last_error = exc

    if attempt % 10 == 0:
        print(
            f"Đang chờ tunnel/DNS... {attempt}s | "
            f"last_error={last_error}"
        )
    time.sleep(1)

if PUBLIC_URL is None:
    print(TUNNEL_LOG.read_text(
        encoding="utf-8", errors="replace"
    )[-12000:])
    raise RuntimeError(
        f"Tunnel chưa sẵn sàng sau 120 giây: {last_error}"
    )

API_BASE_URL = f"{PUBLIC_URL}/api"

print("=" * 72)
print("PUBLIC BACKEND:", PUBLIC_URL)
print("FRONTEND API BASE:", API_BASE_URL)
print()
print(f"VITE_API_BASE_URL={API_BASE_URL}")
print("=" * 72)


## 14. Xác nhận public API

In [ ]:
public_health = requests.get(
    f"{PUBLIC_URL}/api/health",
    timeout=30,
)
print("HTTP status:", public_health.status_code)
print("Response:", public_health.json())
public_health.raise_for_status()


## 15. Kết nối frontend local

Trong:

```text
frontend/.env.local
```

đặt dòng được in ở cell trên:

```env
VITE_API_BASE_URL=https://xxxxx.trycloudflare.com/api
```

Sau đó trên máy local:

```bash
cd frontend
npm install
npm run dev
```

Giữ Colab và cell backend/tunnel đang hoạt động.

Khi frontend bấm **Generate 24h Plan**, backend tiếp tục chạy trên T4. Circuit của run mới tiếp tục được ghi vào:

```text
/content/qaoa-circuits
```

Chạy lại cell **10. Hiển thị circuit** để xem circuit mới.

## 16. Theo dõi GPU và log

In [ ]:
subprocess.run(["nvidia-smi"], check=True)

print("\n--- BACKEND LOG (last 12000 chars) ---\n")
print(BACKEND_LOG.read_text(
    encoding="utf-8",
    errors="replace",
)[-12000:])
